In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw;

In [0]:
%python
#Then create one folder per table:
base = "/Volumes/workspace/bronze/raw"
for t in ["customers","geolocation","order_items","order_payments",
          "order_reviews","orders","products","sellers","category_translation"]:
    dbutils.fs.mkdirs(f"{base}/{t}")

Ingest to Bronze with Auto Loader

In [0]:
%python
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/bronze/raw"
tables = ["customers","geolocation","order_items","order_payments",
          "order_reviews","orders","products","sellers","category_translation"]

def ingest(t):
    opts = {"header": "true", "cloudFiles.format": "csv",
            "cloudFiles.schemaLocation": f"{BASE}/_chk/{t}/schema"}
    if t == "order_reviews":                  # review text has quotes and line breaks
        opts.update({"multiLine": "true", "quote": '"', "escape": '"'})
    (spark.readStream.format("cloudFiles").options(**opts)
        .load(f"{BASE}/{t}/")
        .withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .writeStream
        .option("checkpointLocation", f"{BASE}/_chk/{t}/ckpt")
        .trigger(availableNow=True)
        .toTable(f"workspace.bronze.{t}")
        .awaitTermination())

for t in tables:
    ingest(t)

Check your work

In [0]:
%python
for t in tables:
    print(t, spark.table(f"workspace.bronze.{t}").count())